In [1]:
tabla='ctcam10'
col_essi='fec_cit'
col_tabla='citambproconfec'
essi='essi_dat_cex001_etl'

In [2]:
from decouple import config
import decouple
from sqlalchemy import create_engine
import pandas as pd
import oracledb
import sys
from sqlalchemy import text

In [42]:
#CONEXIONES
config = decouple.AutoConfig(' ')
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [4]:
from datetime import datetime
from sqlalchemy import text


fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=2", con=connection2)
fecha= fecha.iloc[0, 0]
print(fecha)

now = datetime.now()

query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=2"

c1= text(query)
connection2.execute(c1)

2023-02-01


In [5]:
fecha='2023-08-01'

In [6]:

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

base2 = pd.read_sql_query(f"SELECT * FROM {tabla} WHERE {col_tabla} >= TO_DATE('{fecha}', 'YYYY-MM-DD')", con=connection0)


connection0.close()



In [7]:
base2.shape

(151742, 38)

In [8]:
base2 = base2.rename(columns={
    'citambactcod': 'cod_act',
    'citambactespcod': 'cod_sub',
    'citambanufec': 'fec_anu',
    'citambarehoscod': 'cod_are',
    'citambcenasicod': 'cod_cas',
    'citambcnvespcod': 'cnv_esp',
    'citambcrefec': 'fec_cre',
    'citambestctrlseg': 'ctr_seg',
    'citambhorcit': 'hor_cit',
    'citambipanucod': 'ip_anu',
    'citambipcrecod': 'ip_cre',
    'citambipmodcod': 'ip_mod',
    'citambmodanucod': 'mod_anu',
    'citambmodfec': 'fec_mod',
    'citambnum': 'cit_num',
    'citambnumord': 'ord_cit',
    'citamboricenasicod': 'ori_cas',
    'citambpacsecnum': 'pac_sec',
    'citambperasisdocidennum': 'num_doc',
    'citambproconcenasicod': 'cas_pro',
    'citambproconfec': 'fec_cit',
    'citambproconoricenasicod': 'ori_pro',
    'citambproconturhorfin': 'hor_ter',
    'citambproconturhorini': 'hor_ini',
    'citambrep': 'cit_rep',
    'citambservhoscod': 'cod_ser',
    'citambsolfec': 'fec_sol',
    'citambtipdocidenpercod': 'cod_tdi',
    'citambusuanucod': 'usu_anu',
    'citambusucrecod': 'usu_cre',
    'citambusumodcod': 'usu_mod',
    'condcitacod': 'cod_cci',
    'estcitcod': 'cod_eci',
    'estcitotocod': 'cod_enco',
    'modotorcitacod': 'cod_oto',
    'motelicitcod': 'cod_mec',
    'motinacitdes': 'mot_cit',
    'tipocitacod': 'cod_tci'
})

In [9]:
base1=base2

In [10]:
base2=pd.read_sql_query(f"SELECT * FROM {essi} LIMIT 10", con=connection1)

In [11]:
cas = pd.read_sql_query(f"SELECT id_red,cod_cas,des_cas FROM dim_cas where id_red is not null", con=connection2)
red = pd.read_sql_query(f"SELECT id_red,cod_red,des_red FROM dim_red", con=connection2)
cas_red=pd.merge(cas,red,how='left',on='id_red')
base1=pd.merge(base1,cas_red,how='inner',on='cod_cas')

In [12]:
base1.shape

(151742, 42)

In [13]:
serv= pd.read_sql_query(f"SELECT des_ser,cod_ser FROM dim_servicios", con=connection2)
base1=pd.merge(base1,serv,how='inner',on='cod_ser')

In [14]:
base1.shape

(151742, 43)

In [15]:
are = pd.read_sql_query(f"SELECT des_are,cod_are FROM dim_areas", con=connection2)
base1=pd.merge(base1,are,how='inner',on='cod_are')

In [16]:
base1.shape

(151742, 44)

In [17]:

subacti = pd.read_sql_query(f"SELECT des_sub,cod_sub,cod_act FROM dim_subacti", con=connection2)
subacti["KEY"]=subacti["cod_sub"]+subacti["cod_act"]
subacti=subacti.drop(["cod_sub",'cod_act'], axis=1)
base1["KEY"]=base1["cod_sub"].astype(str)+base1['cod_act'].astype(str)
base1["KEY"]=base1["KEY"].str.replace(' ', '', regex=True)
subacti["KEY"]=subacti["KEY"].str.replace(' ', '', regex=True)
base1 = pd.merge(base1,subacti,how='inner',on='KEY')
base1=base1.drop('KEY', axis=1)

In [18]:
base1.shape

(151742, 45)

In [19]:
activi = pd.read_sql_query(f"SELECT cod_act,des_act FROM dim_activi", con=connection2)
base1=pd.merge(base1,activi,how='inner',on='cod_act')

In [20]:
base1.shape

(151742, 46)

In [21]:
tipcita = pd.read_sql_query(f"SELECT cod_tci,des_tci FROM dim_tipcita", con=connection2)
tipcita=tipcita.rename(columns={"cod_tci":"cod_cci"})
base1 = pd.merge(base1,tipcita,how='inner',on='cod_cci')
base1=base1.rename(columns={"des_tci":"des_cci"})

In [22]:
base1.shape

(151742, 47)

In [23]:
cexestsolcit = pd.read_sql_query(f"SELECT cod_esc,des_esc FROM dim_cexestsolcit", con=connection2)
cexestsolcit=cexestsolcit.rename(columns={"cod_esc":"cod_tci"})
base1 = pd.merge(base1,cexestsolcit,how='inner',on='cod_tci')
base1=base1.rename(columns={"des_esc":"des_tci"})

In [24]:
base1.shape

(151742, 48)

In [25]:
cexcitrep = pd.read_sql_query(f"SELECT cod_cre,des_cre FROM dim_cexcitrep", con=connection2)
cexcitrep=cexcitrep.rename(columns={"cod_cre":"cit_rep"})
base1 = pd.merge(base1,cexcitrep,how='inner',on='cit_rep')
base1=base1.rename(columns={"des_cre":"des_rep"})

In [26]:
base1.shape

(151742, 49)

In [27]:
tipemi = pd.read_sql_query(f"SELECT cod_emi,des_emi FROM dim_tipemi", con=connection2)
tipemi=tipemi.rename(columns={"cod_emi":"cod_oto"})
base1 = pd.merge(base1,tipemi,how='inner',on='cod_oto')
base1=base1.rename(columns={"des_emi":"des_oto"})

In [28]:
base1.shape

(151742, 50)

In [29]:
estcit = pd.read_sql_query(f"SELECT cod_eci,des_eci FROM dim_estcit", con=connection2)
base1 = pd.merge(base1,estcit,how='inner',on='cod_eci')


In [30]:
base1.shape

(151742, 51)

In [31]:
cexmotelicit = pd.read_sql_query(f"SELECT cod_eli,des_eli FROM dim_cexmotelicit", con=connection2)
cexmotelicit=cexmotelicit.rename(columns={"cod_eli":"cod_mec"})
base1 = pd.merge(base1,cexmotelicit,how='left',on='cod_mec')
base1=base1.rename(columns={"des_eli":"des_mec"})

In [32]:
cexcitoto = pd.read_sql_query(f"SELECT cod_eco,des_eco FROM dim_cexcitoto", con=connection2)
cexcitoto=cexcitoto.rename(columns={"cod_eco":"cod_enco"})
base1 = pd.merge(base1,cexcitoto,how='inner',on='cod_enco')
base1=base1.rename(columns={"des_eco":"des_enco"})

In [33]:
base1.shape

(151742, 53)

In [34]:
df1_columns = set(base1.columns)
df2_columns = set(base2.columns) 
different_columns = df2_columns - df1_columns
different_columns

set()

In [35]:
borrando=f"DELETE FROM {essi} WHERE {col_essi} >='{fecha}'"
borrado = connection1.execute(borrando)

In [36]:
comunes = set(base1.columns).intersection(set(base2.columns)) 
base = base1[list(comunes)]

In [37]:

base.to_sql(name=f'{essi}', con=connection1, if_exists='append', index=False,chunksize=5000)

30742

In [38]:
#para que continue su camino al dat
base1=base

In [39]:
connection1.close()
connection2.close()
connection3.close()

In [40]:
# Liberar los recursos del motor de base de datos
engine1.dispose()
engine2.dispose()
engine3.dispose()

In [46]:


tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_cit","dt_fecha":"fec_cit","dt_mes":"mes_cit","dt_dia":"dia_cit","dt_dia_sem":"dia_sem_cit","dt_sem":"sem_cit","dt_ano":"ano_cit"})
base1['fec_cit'] = pd.to_datetime(base1['fec_cit']).dt.date
base1=pd.merge(base1,tiempo,how='left',on='fec_cit')

base1['fec_cre_temp']=base1['fec_cre']
base1['fec_mod_temp']=base1['fec_mod']
base1['fec_anu_temp']=base1['fec_anu']
base1['fec_sol_temp']=base1['fec_sol']

tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_sol","dt_fecha":"fec_sol","dt_mes":"mes_sol","dt_dia":"dia_sol","dt_dia_sem":"dia_sem_sol","dt_sem":"sem_sol","dt_ano":"ano_sol"})
base1['fec_sol'] = base1['fec_sol'].astype(str).str[0:10]
base1["fec_sol"]=base1["fec_sol"].str.replace(' ', '', regex=True)
tiempo["fec_sol"]=tiempo["fec_sol"].astype(str).str.replace(' ', '', regex=True)
base1=pd.merge(base1,tiempo,how='left',on='fec_sol')


tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_cre","dt_fecha":"fec_cre","dt_mes":"mes_cre","dt_dia":"dia_cre","dt_dia_sem":"dia_sem_cre","dt_sem":"sem_cre","dt_ano":"ano_cre"})
base1['fec_cre'] = pd.to_datetime(base1['fec_cre']).dt.date
base1=pd.merge(base1,tiempo,how='left',on='fec_cre')

tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_mod","dt_fecha":"fec_mod","dt_mes":"mes_mod","dt_dia":"dia_mod","dt_dia_sem":"dia_sem_mod","dt_sem":"sem_mod","dt_ano":"ano_mod"})
base1['fec_mod'] = pd.to_datetime(base1['fec_mod']).dt.date
base1=pd.merge(base1,tiempo,how='left',on='fec_mod')

tiempo = pd.read_sql_query(f"SELECT id_tiempo,dt_fecha,dt_mes,dt_dia,dt_dia_sem,dt_sem,dt_ano FROM dim_tiempo", con=connection2)
tiempo=tiempo.rename(columns={"id_tiempo":"id_time_anu","dt_fecha":"fec_anu","dt_mes":"mes_anu","dt_dia":"dia_anu","dt_dia_sem":"dia_sem_anu","dt_sem":"sem_anu","dt_ano":"ano_anu"})
base1['fec_anu'] = pd.to_datetime(base1['fec_anu']).dt.date
base1=pd.merge(base1,tiempo,how='left',on='fec_anu')

base1['fec_cre'] = base1['fec_cre_temp']
base1['fec_mod'] = base1['fec_mod_temp']
base1['fec_anu'] = base1['fec_anu_temp']
base1['fec_sol'] = base1['fec_sol_temp']

base1 = base1.drop('fec_cre_temp', axis=1)
base1 = base1.drop('fec_mod_temp', axis=1)
base1 = base1.drop('fec_anu_temp', axis=1)
base1 = base1.drop('fec_sol_temp', axis=1)